# Voice-of-Customer Risk Scorecard

This notebook rebuilds a per-SKU risk scorecard from live PromptathonDb data each time it is run.

## Setup and Environment

Import the libraries needed for live SQL access, parsing, and table display.

In [5]:
%pip install pymssql pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import json
import re
import pandas as pd
import pymssql

try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        print(obj)

    class Markdown(str):
        pass

print('Python packages ready')

Python packages ready


## Data Access Helper

This helper uses the MSSQL connection string from the Codespace environment to query PromptathonDb directly.

In [7]:
def _connection_kwargs_from_env():
    conn_str = os.environ.get('MSSQL_CONNECTION_STRING', '')
    params = {}
    for part in conn_str.split(';'):
        if '=' in part:
            key, value = part.split('=', 1)
            params[key.strip().lower()] = value.strip()

    server = params.get('server', 'sql')
    if ',' in server:
        server, port_str = server.split(',', 1)
        port = int(port_str)
    else:
        port = 1433

    return {
        'server': server,
        'port': port,
        'user': params.get('user id') or params.get('user', 'sa'),
        'password': params.get('password', 'Promptathon!2026'),
        'database': params.get('database', 'PromptathonDb'),
        'as_dict': True,
        'autocommit': True,
    }


def query_promptathon_db(query, params=None):
    kwargs = _connection_kwargs_from_env()
    conn = pymssql.connect(**kwargs)
    try:
        with conn.cursor() as cur:
            cur.execute(query, params or ())
            if cur.description is None:
                return pd.DataFrame()
            columns = [col[0] for col in cur.description]
            rows = cur.fetchall()
        return pd.DataFrame(rows, columns=columns)
    finally:
        conn.close()

print('Helper ready')

Helper ready


## Data Access Smoke Test

The next cell proves live access by returning the Docs row count before any scoring logic runs.

In [8]:
docs_count = query_promptathon_db('SELECT COUNT(*) AS row_count FROM dbo.Docs').iloc[0, 0]
print(f'Live Docs row count: {docs_count}')

Live Docs row count: 89


## Parameters

These values are intentionally exposed so the notebook can be rerun cleanly on new data.

In [9]:
NEGATIVE_RATING_THRESHOLD = 2
RISK_WEIGHTS = {
    'revenue': 0.25,
    'units': 0.10,
    'ticket_count': 0.15,
    'satisfaction': 0.20,
    'negative_reviews': 0.15,
    'negative_docs': 0.15,
}
TOP_N_SKUS = 5

print('Parameters ready')

Parameters ready


## Load and Parse Docs

Pull Docs, parse the TagsJson array into rating/language/SKU values, and flag negative documents.

In [10]:
docs = query_promptathon_db('''
SELECT DocId, SourceType, RelatedSKU, TagsJson, Body
FROM dbo.Docs
ORDER BY DocId
''')

if docs.empty:
    raise RuntimeError('No Docs found from live query')


def parse_tags(tag_json):
    if pd.isna(tag_json):
        return {}
    try:
        tags = json.loads(tag_json)
    except (TypeError, json.JSONDecodeError):
        return {}
    parsed = {}
    for item in tags:
        if not isinstance(item, str):
            continue
        if ':' not in item:
            continue
        key, value = item.split(':', 1)
        parsed[key.strip().lower()] = value.strip()
    return parsed


def parse_doc_row(row):
    parsed = parse_tags(row['TagsJson'])
    rating = None
    if 'rating' in parsed:
        try:
            rating = float(parsed['rating'])
        except ValueError:
            rating = None
    return {
        'rating': rating,
        'language': parsed.get('language'),
        'sku': parsed.get('sku'),
        'scenario': parsed.get('scenario'),
        'resolution': parsed.get('resolution'),
    }

parsed = docs.apply(parse_doc_row, axis=1).apply(pd.Series)
docs = pd.concat([docs.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)
docs['effective_sku'] = docs['sku'].fillna(docs['RelatedSKU'])
docs['negative_review'] = False
docs['negative_doc'] = False

review_mask = (docs['SourceType'].astype(str).str.lower() == 'review') & docs['rating'].notna() & (docs['rating'] <= NEGATIVE_RATING_THRESHOLD)
docs.loc[review_mask, 'negative_review'] = True
docs.loc[review_mask, 'negative_doc'] = True

support_mask = docs['SourceType'].astype(str).str.lower() == 'supportchat'
support_negative_mask = support_mask & (
    docs['scenario'].fillna('').astype(str).str.contains('complaint', case=False, na=False) |
    docs['resolution'].fillna('').astype(str).str.contains('unresolved', case=False, na=False)
)
docs.loc[support_negative_mask, 'negative_doc'] = True

print(docs[['DocId', 'SourceType', 'effective_sku', 'rating', 'scenario', 'resolution', 'negative_review', 'negative_doc']].head(10).to_string(index=False))

 DocId SourceType effective_sku  rating scenario resolution  negative_review  negative_doc
     1     Review ZCETM-SS-L-OB     4.0      NaN        NaN            False         False
     2     Review ZCETM-SS-M-TO     1.0      NaN        NaN             True          True
     3     Review ZCPSW-AS-S-CS     1.0      NaN        NaN             True          True
     4     Review ZCESM-AS-L-CS     5.0      NaN        NaN            False         False
     5     Review ZCPTW-SS-L-BO     3.0      NaN        NaN            False         False
     6     Review ZCPTW-SS-L-CS     2.0      NaN        NaN             True          True
     7     Review ZCPTW-LS-M-BW     1.0      NaN        NaN             True          True
     8     Review ZCPTW-SS-L-BO     2.0      NaN        NaN             True          True
     9     Review ZCETY-SS-L-BO     5.0      NaN        NaN            False         False
    10     Review ZCPPY-AP-L-TO     4.0      NaN        NaN            False         False

## Load Support Tickets and SalesOrderLines

Pull support-ticket and line-item data from PromptathonDb and compute the required fields.

In [11]:
support_tickets = query_promptathon_db('''
SELECT RelatedSKU, Priority, Status, SatisfactionScore
FROM dbo.SupportTickets
WHERE RelatedSKU IS NOT NULL
ORDER BY RelatedSKU
''')

sales_order_lines = query_promptathon_db('''
SELECT SKU, Quantity, LineTotal
FROM dbo.SalesOrderLines
WHERE SKU IS NOT NULL
ORDER BY SKU
''')

support_tickets['SatisfactionScore'] = pd.to_numeric(support_tickets['SatisfactionScore'], errors='coerce')
sales_order_lines['Quantity'] = pd.to_numeric(sales_order_lines['Quantity'], errors='coerce')
sales_order_lines['LineTotal'] = pd.to_numeric(sales_order_lines['LineTotal'], errors='coerce')

print(f'Support tickets rows: {len(support_tickets)}')
print(f'Sales order lines rows: {len(sales_order_lines)}')

Support tickets rows: 33
Sales order lines rows: 2728


## Aggregate SKU Metrics

Aggregate revenue, units, ticket count, average satisfaction score, and negative counts per SKU.

In [12]:
sku_revenue_units = (
    sales_order_lines.groupby('SKU', as_index=False)
    .agg(revenue=('LineTotal', 'sum'), units=('Quantity', 'sum'))
)

sku_ticket_metrics = (
    support_tickets.groupby('RelatedSKU', as_index=False)
    .agg(
        ticket_count=('RelatedSKU', 'size'),
        avg_satisfaction_score=('SatisfactionScore', 'mean'),
    )
)

sku_doc_metrics = (
    docs.groupby('effective_sku', as_index=False)
    .agg(
        negative_review_count=('negative_review', 'sum'),
        negative_doc_count=('negative_doc', 'sum'),
    )
)

score_input = (
    sku_revenue_units.merge(sku_ticket_metrics, left_on='SKU', right_on='RelatedSKU', how='left')
    .merge(sku_doc_metrics, left_on='SKU', right_on='effective_sku', how='left')
    .drop(columns=['RelatedSKU', 'effective_sku'])
)

score_input['ticket_count'] = score_input['ticket_count'].fillna(0).astype(int)
score_input['avg_satisfaction_score'] = pd.to_numeric(score_input['avg_satisfaction_score'], errors='coerce')
score_input['negative_review_count'] = score_input['negative_review_count'].fillna(0).astype(int)
score_input['negative_doc_count'] = score_input['negative_doc_count'].fillna(0).astype(int)
score_input['revenue'] = pd.to_numeric(score_input['revenue'], errors='coerce').fillna(0.0)
score_input['units'] = pd.to_numeric(score_input['units'], errors='coerce').fillna(0.0)

print(score_input.head(10).to_string(index=False))

          SKU  revenue  units  ticket_count  avg_satisfaction_score  negative_review_count  negative_doc_count
ZCEPM-AP-L-BO  1478.27     17             0                     NaN                      0                   0
ZCEPM-AP-L-BW  3309.22     38             0                     NaN                      0                   0
ZCEPM-AP-L-OB   839.32     10             0                     NaN                      0                   0
ZCEPM-AP-L-TO   712.00      8             0                     NaN                      0                   0
ZCEPM-AP-M-BW  1142.54     13             0                     NaN                      0                   0
ZCEPM-AP-M-CS   861.14     10             0                     NaN                      0                   0
ZCEPM-AP-M-TO    73.95      1             0                     NaN                      0                   0
ZCEPM-AP-S-BW  1602.00     18             0                     NaN                      0                   0
Z

## Compute Transparent Risk Score

Normalize the metric columns with min-max scaling and combine them with the documented weights.

In [13]:
def minmax(series):
    series = pd.to_numeric(series, errors='coerce').fillna(0.0)
    if series.nunique() <= 1:
        return pd.Series(0.0, index=series.index)
    mn = series.min()
    mx = series.max()
    return (series - mn) / (mx - mn)

score_input['revenue_risk'] = minmax(score_input['revenue'])
score_input['units_risk'] = minmax(score_input['units'])
score_input['ticket_risk'] = minmax(score_input['ticket_count'])

satisfaction_series = score_input['avg_satisfaction_score'].fillna(score_input['avg_satisfaction_score'].median())
score_input['satisfaction_risk'] = 1 - minmax(satisfaction_series)
score_input['negative_review_risk'] = minmax(score_input['negative_review_count'])
score_input['negative_doc_risk'] = minmax(score_input['negative_doc_count'])

score_input['risk_score'] = (
    score_input['revenue_risk'] * RISK_WEIGHTS['revenue'] +
    score_input['units_risk'] * RISK_WEIGHTS['units'] +
    score_input['ticket_risk'] * RISK_WEIGHTS['ticket_count'] +
    score_input['satisfaction_risk'] * RISK_WEIGHTS['satisfaction'] +
    score_input['negative_review_risk'] * RISK_WEIGHTS['negative_reviews'] +
    score_input['negative_doc_risk'] * RISK_WEIGHTS['negative_docs']
)

scorecard = score_input.sort_values('risk_score', ascending=False).reset_index(drop=True)
scorecard['risk_score'] = scorecard['risk_score'].round(4)

scorecard[['SKU', 'revenue', 'units', 'ticket_count', 'avg_satisfaction_score', 'negative_review_count', 'negative_doc_count', 'risk_score']].head(10)

,SKU,revenue,units,ticket_count,avg_satisfaction_score,negative_review_count,negative_doc_count,risk_score
0,ZCPTM-SS-M-BW,19825.05,228,9,1.666667,6,6,0.9667
1,ZCETM-LS-L-BO,18297.21,213,1,3.000000,0,1,0.4657
2,ZCETM-LS-M-BO,19149.86,222,0,NaN,0,0,0.3888
3,ZCETM-SS-L-BO,12363.65,143,2,3.500000,0,1,0.3515
4,ZCPTW-SS-L-BO,4305.87,50,1,1.000000,1,1,0.3418
5,ZCETM-SS-L-OB,16026.57,185,0,NaN,0,0,0.3330
6,ZCETM-SS-S-BO,9583.90,110,3,3.333333,0,0,0.3017
7,ZCPTM-SS-L-BW,13289.22,154,1,4.000000,0,0,0.3013
8,ZCETW-SS-L-BW,11367.16,132,1,4.000000,0,0,0.2673
9,ZCPTM-LS-L-BO,10427.08,121,0,NaN,0,0,0.2339


## Generate Ranked Scorecard and Summary

Display the ranked scorecard and a short markdown summary of the most at-risk SKUs.

In [14]:
ranked_scorecard = scorecard.head(TOP_N_SKUS).copy()
display(ranked_scorecard[['SKU', 'revenue', 'units', 'ticket_count', 'avg_satisfaction_score', 'negative_review_count', 'negative_doc_count', 'risk_score']])

summary_lines = [
    f'Live scorecard generated from PromptathonDb on {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}.',
    f'Top {TOP_N_SKUS} highest-risk SKUs:'
]
for _, row in ranked_scorecard.iterrows():
    summary_lines.append(
        f"- {row['SKU']}: risk {row['risk_score']:.3f}, revenue ${row['revenue']:,.0f}, units {int(row['units'])}, tickets {int(row['ticket_count'])}, negative reviews {int(row['negative_review_count'])}, negative docs {int(row['negative_doc_count'])}"
    )

display(Markdown('\n'.join(summary_lines)))

,SKU,revenue,units,ticket_count,avg_satisfaction_score,negative_review_count,negative_doc_count,risk_score
0,ZCPTM-SS-M-BW,19825.05,228,9,1.666667,6,6,0.9667
1,ZCETM-LS-L-BO,18297.21,213,1,3.000000,0,1,0.4657
2,ZCETM-LS-M-BO,19149.86,222,0,NaN,0,0,0.3888
3,ZCETM-SS-L-BO,12363.65,143,2,3.500000,0,1,0.3515
4,ZCPTW-SS-L-BO,4305.87,50,1,1.000000,1,1,0.3418


Live scorecard generated from PromptathonDb on 2026-07-10 17:10:30.
Top 5 highest-risk SKUs:
- ZCPTM-SS-M-BW: risk 0.967, revenue $19,825, units 228, tickets 9, negative reviews 6, negative docs 6
- ZCETM-LS-L-BO: risk 0.466, revenue $18,297, units 213, tickets 1, negative reviews 0, negative docs 1
- ZCETM-LS-M-BO: risk 0.389, revenue $19,150, units 222, tickets 0, negative reviews 0, negative docs 0
- ZCETM-SS-L-BO: risk 0.351, revenue $12,364, units 143, tickets 2, negative reviews 0, negative docs 1
- ZCPTW-SS-L-BO: risk 0.342, revenue $4,306, units 50, tickets 1, negative reviews 1, negative docs 1

## Optional Similarity Analysis for the Top Risk SKU

For the highest-risk SKU, pull one negative review and run the built-in similarity procedure as supporting evidence.

In [15]:
if not ranked_scorecard.empty:
    top_sku = ranked_scorecard.iloc[0]['SKU']
    candidate_reviews = docs[(docs['effective_sku'] == top_sku) & docs['negative_review'] & (docs['SourceType'].astype(str).str.lower() == 'review')]

    if candidate_reviews.empty:
        print(f'No negative review docs found for {top_sku}; similarity analysis skipped.')
    else:
        sample_doc = candidate_reviews.iloc[0]
        similar_df = query_promptathon_db(
            'EXEC dbo.FindSimilarDocsByDocId @DocId=%s, @TopN=%s',
            (int(sample_doc['DocId']), 5)
        )
        similar_df = similar_df[['DocId', 'Distance']].copy()
        similar_df = similar_df.sort_values('Distance').reset_index(drop=True)
        display(Markdown(f'Similarity evidence for {top_sku} using negative review doc {int(sample_doc["DocId"])} (lower cosine distance = closer).'))
        display(similar_df)
else:
    print('No ranked SKUs available for similarity analysis.')

Similarity evidence for ZCPTM-SS-M-BW using negative review doc 39 (lower cosine distance = closer).

,DocId,Distance
0,39,0.000000
1,44,0.211081
2,42,0.227228
3,40,0.236689
4,41,0.252660
